### Logic

In [ ]:
"""
Utility to convert methylation prediction DataFrame to BAM format for IGV visualization.

This creates BAM files where:
1. Methylation is displayed ON each read (using MM/ML tags)
2. Reads can be colored/sorted by prediction category (using RG tags)

Requirements:
    pip install pysam pandas numpy

Usage:
    from methylation_to_bam import export_methylation_bam

    files = export_methylation_bam(
        df,
        output_dir='./igv_output',
        prefix='my_sample',
        reference_fasta='hg38.fa',  # Optional but recommended
        reject_class=39
    )
"""

import pandas as pd
import numpy as np
from pathlib import Path
from typing import Dict, List, Optional, Tuple
from collections import defaultdict
import re
import array

# =============================================================================
# PREDICTION CLASSIFICATION
# =============================================================================

PREDICTION_CATEGORIES = {
    "correctly_predicted": {
        "id": "CP",
        "description": "Correctly Predicted (label == prediction, label != reject)",
        "color": "34,139,34",  # Forest Green
    },
    "incorrectly_rejected": {
        "id": "IR",
        "description": "Incorrectly Rejected (label != reject, prediction == reject)",
        "color": "255,140,0",  # Dark Orange
    },
    "incorrectly_predicted": {
        "id": "IP",
        "description": "Incorrectly Predicted (label == reject, prediction != reject)",
        "color": "220,20,60",  # Crimson
    },
    "correctly_rejected": {
        "id": "CR",
        "description": "Correctly Rejected (label == prediction == reject)",
        "color": "128,128,128",  # Gray
    },
}


def classify_prediction(label: int, prediction: int, reject_class: int = 39) -> str:
    """Classify prediction into one of four categories."""
    if label == prediction and label != reject_class:
        return "correctly_predicted"
    elif label != reject_class and prediction == reject_class:
        return "incorrectly_rejected"
    elif label == reject_class and prediction != reject_class:
        return "incorrectly_predicted"
    elif label == prediction == reject_class:
        return "correctly_rejected"
    return "correctly_predicted"  # Default fallback


# =============================================================================
# MM/ML TAG GENERATION FOR METHYLATION
# =============================================================================


def parse_methylation_to_mm_ml(
    sequence: str, methylation_ids: str, strand: str = "+"
) -> Tuple[Optional[str], Optional[array.array]]:
    """
    Convert methylation string to MM and ML tags for BAM.

    MM tag format: "C+m,pos1,pos2,..." or "C+m?,pos1,pos2,..."
        - Lists positions of modified cytosines (as deltas between C positions)
    ML tag format: Array of probabilities (0-255) for each modification

    Parameters
    ----------
    sequence : str
        DNA sequence of the read
    methylation_ids : str
        String of 0/1/2 for each position (0=unmethylated, 1=methylated, 2=not CpG)
    strand : str
        Strand orientation ('+' or '-')

    Returns
    -------
    Tuple[str, bytes]
        MM tag string and ML tag byte array, or (None, None) if no CpGs
    """
    if len(sequence) != len(methylation_ids):
        # Truncate or pad to match
        min_len = min(len(sequence), len(methylation_ids))
        sequence = sequence[:min_len]
        methylation_ids = methylation_ids[:min_len]

    # Find all C positions in the sequence
    c_positions = []
    for i, base in enumerate(sequence.upper()):
        if base == "C":
            c_positions.append(i)

    if not c_positions:
        return None, None

    # For each C, check if it has methylation info (0 or 1, not 2)
    # MM tag uses delta encoding: distance from previous C to next modified C
    mm_deltas = []
    ml_probs = []

    skipped_cs = 0  # Count of Cs skipped since last reported one

    for c_pos in c_positions:
        if c_pos < len(methylation_ids):
            methyl_state = methylation_ids[c_pos]

            if methyl_state in ("0", "1"):
                # This is a CpG site with known methylation
                mm_deltas.append(str(skipped_cs))
                skipped_cs = 0

                # ML probability: 255 = methylated, 0 = unmethylated
                if methyl_state == "1":
                    ml_probs.append(255)  # Methylated
                else:
                    ml_probs.append(0)  # Unmethylated
            else:
                # State is '2' (unknown/not CpG), skip this C
                skipped_cs += 1
        else:
            skipped_cs += 1

    if not mm_deltas:
        return None, None

    # Format MM tag: "C+m?,delta1,delta2,..."
    # The '?' indicates that untagged Cs have unknown status
    mm_tag = "C+m?," + ",".join(mm_deltas) + ";"

    # ML tag must be an array of unsigned bytes for BAM format
    ml_tag = array.array("B", ml_probs)

    return mm_tag, ml_tag


def parse_methylation_to_mm_ml_v2(
    sequence: str, methylation_ids: str, strand: str = "+"
) -> Tuple[Optional[str], Optional[array.array]]:
    """
    Alternative simpler approach: Report ALL CpGs (both methylated and unmethylated).

    This version marks every position where we have methylation data,
    allowing IGV to show both states clearly.
    """
    if len(sequence) != len(methylation_ids):
        min_len = min(len(sequence), len(methylation_ids))
        sequence = sequence[:min_len]
        methylation_ids = methylation_ids[:min_len]

    # Find CpG sites (positions where methylation_ids is 0 or 1)
    cpg_positions = []
    ml_probs = []

    for i, state in enumerate(methylation_ids):
        if state in ("0", "1"):
            cpg_positions.append(i)
            ml_probs.append(255 if state == "1" else 0)

    if not cpg_positions:
        return None, None

    # Find all C positions to calculate deltas
    c_positions = [i for i, base in enumerate(sequence.upper()) if base == "C"]

    if not c_positions:
        return None, None

    # Calculate delta encoding
    mm_deltas = []
    c_idx = 0

    for cpg_pos in cpg_positions:
        # Count how many Cs we skip to reach this CpG
        skipped = 0
        while c_idx < len(c_positions) and c_positions[c_idx] < cpg_pos:
            skipped += 1
            c_idx += 1

        if c_idx < len(c_positions) and c_positions[c_idx] == cpg_pos:
            mm_deltas.append(str(skipped))
            c_idx += 1
        else:
            # CpG position doesn't match a C in sequence - use absolute position approach
            mm_deltas.append(str(skipped))

    if not mm_deltas:
        return None, None

    mm_tag = "C+m?," + ",".join(mm_deltas) + ";"
    ml_tag = array.array("B", ml_probs)

    return mm_tag, ml_tag


# =============================================================================
# BAM FILE CREATION
# =============================================================================


def create_bam_with_methylation(
    df: pd.DataFrame,
    output_path: str,
    reference_fasta: Optional[str] = None,
    reject_class: int = 39,
) -> str:
    """
    Create a BAM file with methylation tags and read group annotations.

    Parameters
    ----------
    df : pd.DataFrame
        Input dataframe with read-level data
    output_path : str
        Path to save the BAM file
    reference_fasta : str, optional
        Path to reference FASTA (for proper header generation)
    reject_class : int
        Class label representing "rejected" predictions

    Returns
    -------
    str
        Path to the created BAM file
    """
    try:
        import pysam
    except ImportError:
        raise ImportError("pysam is required. Install with: pip install pysam")

    df = df.copy()

    # Classify predictions
    df["category"] = df.apply(
        lambda row: classify_prediction(row["label"], row["prediction"], reject_class),
        axis=1,
    )

    # Get unique chromosomes and calculate their lengths
    df["read_end"] = df["ref_pos"] + df["read_length"]
    chrom_lengths = df.groupby("chromosome")["read_end"].max().to_dict()

    # Sort chromosomes naturally
    def natural_sort_key(s):
        return [int(t) if t.isdigit() else t.lower() for t in re.split(r"(\d+)", s)]

    sorted_chroms = sorted(chrom_lengths.keys(), key=natural_sort_key)

    # Create header
    header = {
        "HD": {"VN": "1.6", "SO": "coordinate"},
        "SQ": [
            {"SN": chrom, "LN": int(chrom_lengths[chrom] + 1000)}
            for chrom in sorted_chroms
        ],
        "RG": [
            {
                "ID": info["id"],
                "SM": "sample",
                "DS": info["description"],
                "PL": "ILLUMINA",
            }
            for cat, info in PREDICTION_CATEGORIES.items()
        ],
    }

    # Create BAM file
    with pysam.AlignmentFile(output_path, "wb", header=header) as bam:

        # Create chromosome name to index mapping
        chrom_to_idx = {chrom: idx for idx, chrom in enumerate(sorted_chroms)}

        # Sort dataframe by chromosome and position
        df["_chrom_idx"] = df["chromosome"].map(chrom_to_idx)
        df = df.sort_values(["_chrom_idx", "ref_pos"]).reset_index(drop=True)

        for idx, row in df.iterrows():
            # Create alignment segment
            a = pysam.AlignedSegment()
            a.query_name = f"read_{row['read_id']}"
            a.query_sequence = row["input_ids"]
            a.flag = 0  # Forward strand, mapped
            a.reference_id = chrom_to_idx[row["chromosome"]]
            a.reference_start = row["ref_pos"]
            a.mapping_quality = 60

            # CIGAR: all matches for simplicity
            a.cigar = [(0, len(row["input_ids"]))]  # 0 = M (match/mismatch)

            # Quality scores (dummy high quality)
            a.query_qualities = pysam.qualitystring_to_array(
                "I" * len(row["input_ids"])
            )

            # Add read group tag for prediction category
            category = row["category"]
            rg_id = PREDICTION_CATEGORIES[category]["id"]
            a.set_tag("RG", rg_id)

            # Add custom tags for additional info
            a.set_tag("XC", category[:20])  # Category name (truncated)
            a.set_tag("XL", row["label"])  # True label
            a.set_tag("XP", row["prediction"])  # Prediction
            a.set_tag("XF", row["confidence"])  # Confidence

            # Add methylation tags (MM and ML)
            mm_tag, ml_tag = parse_methylation_to_mm_ml(
                row["input_ids"], str(row["methylation_ids"])
            )

            if mm_tag and ml_tag:
                a.set_tag("MM", mm_tag)
                a.set_tag("ML", ml_tag)

            bam.write(a)

    # Sort and index the BAM file
    sorted_path = output_path.replace(".bam", ".sorted.bam")
    pysam.sort("-o", sorted_path, output_path)
    pysam.index(sorted_path)

    # Remove unsorted file
    Path(output_path).unlink()

    # Rename sorted to original name
    Path(sorted_path).rename(output_path)
    Path(sorted_path + ".bai").rename(output_path + ".bai")

    print(f"Created BAM file: {output_path}")
    print(f"Created BAM index: {output_path}.bai")

    # Print category summary
    category_counts = df["category"].value_counts()
    print(f"\nRead categories:")
    for cat, count in category_counts.items():
        print(f"  {cat}: {count}")

    return output_path


# =============================================================================
# ALTERNATIVE: SEPARATE BAM FILES PER CATEGORY
# =============================================================================


def create_bams_per_category(
    df: pd.DataFrame,
    output_dir: str,
    prefix: str = "methylation",
    reject_class: int = 39,
) -> Dict[str, str]:
    """
    Create separate BAM files for each prediction category.

    This allows loading each category as a separate track in IGV,
    making it easier to visually compare categories.

    Parameters
    ----------
    df : pd.DataFrame
        Input dataframe
    output_dir : str
        Output directory
    prefix : str
        Prefix for output files
    reject_class : int
        Reject class label

    Returns
    -------
    Dict[str, str]
        Mapping of category to BAM file path
    """
    try:
        import pysam
    except ImportError:
        raise ImportError("pysam is required. Install with: pip install pysam")

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    df = df.copy()
    df["category"] = df.apply(
        lambda row: classify_prediction(row["label"], row["prediction"], reject_class),
        axis=1,
    )

    files = {}

    for category in df["category"].unique():
        category_df = df[df["category"] == category]
        cat_short = PREDICTION_CATEGORIES.get(category, {}).get(
            "id", category[:2].upper()
        )
        output_path = str(output_dir / f"{prefix}_{cat_short}.bam")

        create_bam_with_methylation(category_df, output_path, reject_class=reject_class)

        files[category] = output_path

    return files


# =============================================================================
# DMR BED FILE (Separate track)
# =============================================================================


def create_dmr_bed(df: pd.DataFrame, output_path: str) -> str:
    """Create a BED file for DMR regions as a separate track."""

    df = df.copy()
    df["read_end"] = df["ref_pos"] + df["read_length"]

    # Aggregate DMRs
    dmr_agg = (
        df.groupby(["dmr_label", "chromosome"])
        .agg(
            start=("ref_pos", "min"),
            end=("read_end", "max"),
            dmr_ctype=("dmr_ctype", "first"),
            n_reads=("read_id", "count"),
        )
        .reset_index()
    )

    bed_df = pd.DataFrame(
        {
            "chrom": dmr_agg["chromosome"],
            "chromStart": dmr_agg["start"],
            "chromEnd": dmr_agg["end"],
            "name": dmr_agg.apply(
                lambda r: f"DMR_{r['dmr_label']}|{r['dmr_ctype']}", axis=1
            ),
            "score": 0,
            "strand": ".",
        }
    )

    bed_df = bed_df.sort_values(["chrom", "chromStart"])
    bed_df.to_csv(output_path, sep="\t", header=False, index=False)

    print(f"Created DMR BED: {output_path}")
    return output_path


# =============================================================================
# READ GROUP COLOR FILE FOR IGV
# =============================================================================


def create_read_group_colors(output_path: str) -> str:
    """
    Create a read group color file for IGV.

    This file tells IGV what color to use for each read group.
    In IGV: View > Preferences > Alignments > Color by: read group

    Format: tab-separated, RG_ID <tab> R,G,B
    """
    lines = []
    for cat, info in PREDICTION_CATEGORIES.items():
        lines.append(f"{info['id']}\t{info['color']}")

    with open(output_path, "w") as f:
        f.write("\n".join(lines))

    print(f"Created read group colors: {output_path}")
    print("  To use in IGV:")
    print("  1. Right-click on track > Color alignments by > read group")
    print(
        f"  2. Load this file via View > Preferences > Alignments > Read group colors"
    )

    return output_path


# =============================================================================
# IGV SESSION FILE
# =============================================================================


def create_igv_session(
    bam_path: str,
    dmr_path: Optional[str],
    output_path: str,
    genome: str = "hg38",
    locus: Optional[str] = None,
) -> str:
    """Create an IGV session XML file."""

    bam_abs = str(Path(bam_path).absolute())

    resources = [f'        <Resource path="{bam_abs}"/>']

    if dmr_path:
        dmr_abs = str(Path(dmr_path).absolute())
        resources.append(f'        <Resource path="{dmr_abs}"/>')

    resources_str = "\n".join(resources)

    locus_attr = f'locus="{locus}"' if locus else ""

    xml = f"""<?xml version="1.0" encoding="UTF-8" standalone="no"?>
<Session genome="{genome}" hasGeneTrack="true" hasSequenceTrack="true" {locus_attr} version="8">
    <Resources>
{resources_str}
    </Resources>
    <Panel name="DataPanel">
        <Track id="{bam_abs}" name="Reads with Methylation">
            <RenderOptions colorOption="READ_GROUP"/>
        </Track>
    </Panel>
</Session>
"""

    with open(output_path, "w") as f:
        f.write(xml)

    print(f"Created IGV session: {output_path}")
    return output_path


# =============================================================================
# MAIN EXPORT FUNCTION
# =============================================================================


def export_methylation_bam(
    df: pd.DataFrame,
    output_dir: str,
    prefix: str = "methylation",
    reject_class: int = 39,
    separate_categories: bool = False,
    genome: str = "hg38",
) -> Dict[str, str]:
    """
    Export methylation data to BAM format for IGV visualization.

    Creates:
    1. BAM file(s) with methylation tags (MM/ML) and read groups (RG)
    2. DMR regions as BED file
    3. Read group color file for IGV
    4. IGV session file

    Parameters
    ----------
    df : pd.DataFrame
        Input dataframe with read-level predictions and methylation
    output_dir : str
        Output directory
    prefix : str
        Prefix for output filenames
    reject_class : int
        Class label for "reject" predictions (default: 39)
    separate_categories : bool
        If True, create separate BAM files per prediction category
    genome : str
        Reference genome for IGV session (default: hg38)

    Returns
    -------
    Dict[str, str]
        Dictionary mapping file type to file path
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    files = {}

    print("=" * 70)
    print("Exporting methylation data to BAM format for IGV")
    print("=" * 70)
    print()

    # 1. Create BAM file(s)
    if separate_categories:
        bam_files = create_bams_per_category(df, str(output_dir), prefix, reject_class)
        files.update({f"bam_{cat}": path for cat, path in bam_files.items()})
        main_bam = list(bam_files.values())[0]  # Use first for session
    else:
        bam_path = str(output_dir / f"{prefix}.bam")
        create_bam_with_methylation(df, bam_path, reject_class=reject_class)
        files["bam"] = bam_path
        main_bam = bam_path

    print()

    # 2. Create DMR BED file
    dmr_path = str(output_dir / f"{prefix}_dmrs.bed")
    create_dmr_bed(df, dmr_path)
    files["dmrs"] = dmr_path
    print()

    # 3. Create read group color file
    colors_path = str(output_dir / f"{prefix}_rg_colors.txt")
    create_read_group_colors(colors_path)
    files["colors"] = colors_path
    print()

    # 4. Create IGV session
    session_path = str(output_dir / f"{prefix}_session.xml")

    # Get a representative locus
    sample_row = df.iloc[0]
    locus = f"{sample_row['chromosome']}:{sample_row['ref_pos']}-{sample_row['ref_pos'] + 1000}"

    create_igv_session(main_bam, dmr_path, session_path, genome, locus)
    files["session"] = session_path

    # Print instructions
    print()
    print("=" * 70)
    print("HOW TO VIEW IN IGV")
    print("=" * 70)
    print("""
1. Open IGV (version 2.12+ recommended for MM/ML tag support)

2. Load your data:
   - File > Open Session > {session_path}
   OR
   - File > Load from File > {bam_path}

3. Configure methylation visualization:
   - Right-click on the track
   - Select "Color alignments by" > "base modification (5mC)"
   
4. Configure prediction category colors:
   - Right-click on the track  
   - Select "Color alignments by" > "read group"
   - (Colors: Green=Correct, Orange=IncorrectlyRejected, Red=IncorrectlyPredicted, Gray=CorrectlyRejected)

5. Sort reads by category:
   - Right-click on the track
   - Select "Sort alignments by" > "read group"
   
6. Group reads by category:
   - Right-click on the track
   - Select "Group alignments by" > "read group"

COLOR LEGEND (Prediction Categories):
""".format(session_path=files.get("session", "N/A"), bam_path=main_bam))

    for cat, info in PREDICTION_CATEGORIES.items():
        print(f"  {info['id']}: {cat} - RGB({info['color']})")

    print("""
METHYLATION COLORS (when colored by base modification):
  Red:  Methylated CpG
  Blue: Unmethylated CpG
""")

    return files

### Usage

In [ ]:
import pickle
import pandas as pd

In [ ]:
with open(
    "../Tutorials/methylBertLoyferWithReject_attentionClassifier_DMRs_stratified_v2/predictions_test.pkl",
    "rb",
) as f:
    res_attention = pickle.load(f)

predictions = np.argmax(res_attention[0], 1)
labels = res_attention[1]
confidence = res_attention[0].max(axis=1)

data_path = "/home/luna.kuleuven.be/u0169940/Data/Loyfer/TrainingDataWithRejectionDMRsStratified/"
all_data = []
for split in ["train", "valid", "test"]:
    x = pd.read_parquet(data_path + f"/{split}.parquet")
    x["split"] = split
    all_data.append(x)

all_data = pd.concat(all_data)

test_data = all_data[all_data["split"] == "test"]

test_data["methylated_CpGs"] = test_data["original_methyl"].apply(
    lambda x: x.count("C")
)
test_data["unmethylated_CpGs"] = test_data["original_methyl"].apply(
    lambda x: x.count("T")
)
test_data["methylation_level"] = test_data["methylated_CpGs"] / (
    test_data["methylated_CpGs"] + test_data["unmethylated_CpGs"]
)

test_data["prediction"] = predictions
test_data["confidence"] = confidence

In [ ]:
files = export_methylation_bam(
    test_data,
    output_dir="../../../IGV/Loyfer",
    prefix="loyfer_methylbert",
    reject_class=39,
    separate_categories=True,
)